In [32]:
import numpy as np
# import pandas as pd
import cv2
# import matplotlib.pyplot as plt
import os
# import torch
# import torchvision

In [23]:
# Define paths
train_images_folder_path = "../data/train/images" 
train_mask_folder_path = "../data/train/masks" 
test_images_folder_path = "../data/val/images" 
test_mask_folder_path = "../data/val/masks" 

In [24]:
# Get image and mask names
train_images_names_original = sorted([img for img in os.listdir(train_images_folder_path) if img.endswith('.png')])
train_mask_names_original = sorted([img for img in os.listdir(train_mask_folder_path) if img.endswith('.png')])
test_images_names = sorted([img for img in os.listdir(test_images_folder_path) if img.endswith('.png')])
test_mask_names = sorted([img for img in os.listdir(test_mask_folder_path) if img.endswith('.png')])

In [ ]:
# class SegmentationDataset(Dataset):
#     def __init__(self, image_paths, mask_paths, transform=None):
#         self.image_paths = image_paths
#         self.mask_paths = mask_paths
        
#         # --- تعریف Transform برای Augmentation ---
#         self.augment = Compose([
#             # 1. Geometric Transformations (برای هر دو اعمال شود)
#             Resize(height=H, width=W),
#             Rotate(limit=20, p=0.7),
#             HorizontalFlip(p=0.5),
#             ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.4),

#             # 2. Photometric Transformations (فقط برای Image اعمال شود)
#             RandomBrightnessContrast(p=0.5, brightness_limit=0.2, contrast_limit=0.2),
#             GaussNoise(var_limit=(10.0, 30.0), p=0.3),
            
#             # 3. تبدیل نهایی به تنسور (این باید در آخر باشد)
#             ToTensorV2()
#         ])
        
#         # اگر در زمان تست transform نخواستید، این را جداگانه تعریف کنید
#         self.test_transform = Compose([
#             Resize(height=H, width=W),
#             ToTensorV2()
#         ])
        
#         self.use_augment = transform # این پارامتر باید True باشد
        
#     def __len__(self):
#         return len(self.image_paths)

#     def __getitem__(self, idx):
#         img_path = self.image_paths[idx]
#         mask_path = self.mask_paths[idx]

#         # خواندن تصویر و ماسک
#         # توجه: masks باید حتماً تک کاناله (CV_8UC1) خوانده شوند.
#         image = cv2.imread(img_path)
#         mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
#         # تبدیل رنگ از BGR به RGB ( استاندارد برای مدل‌های DL)
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

#         # --- اعمال تبدیل‌ها ---
#         if self.use_augment:
#             # Albumentations اجازه می‌دهد همزمان روی image و mask کار کنیم.
#             # مهم: اگر ماسک شما مالتی‌کلاس است، باید مطمئن شوید که این ماسک‌ها به درستی 
#             # توسط transform هندل می‌شوند (معمولاً با استفاده از A.Masked 
#             # در نسخه‌های جدیدتر یا تنظیم پارامترها).
            
#             transformed = self.augment(image=image, mask=mask)
#             final_image = transformed['image']
#             final_mask = transformed['mask']
            
#         else:
#             # در زمان تست/اعتبارسنجی (بدون Augmentation)
#             transformed = self.test_transform(image=image, mask=mask)
#             final_image = transformed['image']
#             final_mask = transformed['mask']
        
#         # در نهایت، تبدیل ماسک به تنسور (اگر ToTensorV2 اعمال شده باشد، این انجام شده است)
#         # برای اطمینان، تبدیل دستی به تنسور PyTorch (به جای ToTensorV2 در صورت نیاز)
#         # return final_image, torch.from_numpy(final_mask).long() 
        
#         # با فرض استفاده از ToTensorV2 که تنسور را خروجی می‌دهد:
#         return final_image, final_mask


In [25]:
def one_hot_mask(mask):
    """Convert RGB mask to one-hot encoded tensor."""
    mask = np.array(mask)
    one_hot_map = []
    for color in colors:
        class_map = np.all(mask == color, axis=-1)
        one_hot_map.append(class_map)
    one_hot = np.stack(one_hot_map, axis=-1)
    return torch.from_numpy(one_hot.astype(np.int32)).permute(2, 0, 1)


In [26]:
def preprocess_image(img_path, height=96, width=256, normalize=True, crop=False):
    """Load and preprocess image."""
    img = Image.open(img_path).convert("RGB")

    if crop:
        img = img.crop((0, 0, 2048, 768))  
        
    img = img.resize((width, height))
    img = F.to_tensor(img)

    if normalize:  
        img = F.normalize(img, mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
    return img


In [27]:
def preprocess_mask(mask_path, height=96, width=256, one_hot=True, crop=False):
    """Load and preprocess mask."""
    mask = Image.open(mask_path).convert("RGB")

    if crop:
        mask = mask.crop((0, 0, 2048, 768))

    mask = mask.resize((width, height), resample=Image.NEAREST)

    if one_hot:
        mask = one_hot_mask(mask)
    else:
        mask = F.to_tensor(mask)

    return mask


In [28]:
def augment_image_mask(image, mask):
    """Apply same augmentation to both image and mask."""
    if torch.rand(1) < 0.5:
        image = F.hflip(image)
        mask = F.hflip(mask)

    angle = transforms.RandomRotation.get_params([-15, 15])
    image = F.rotate(image, angle)
    mask = F.rotate(mask, angle)

    jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
    image = jitter(image)

    return image, mask


In [29]:
def load_train(img_folder, mask_folder, img_name, mask_name):
    """Load train image and mask with augmentation pipeline."""
    img_path = os.path.join(img_folder, img_name)
    mask_path = os.path.join(mask_folder, mask_name)

    image = preprocess_image(img_path)
    mask = preprocess_mask(mask_path)

    image, mask = augment_image_mask(image, mask)
    return image, mask
